## Initialize the SDK

In [1]:
import goodfire
import os
    
GOODFIRE_API_KEY = os.getenv("GOODFIRE_API_KEY")
client = goodfire.Client(GOODFIRE_API_KEY)

# Instantiate a model variant
variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

In [2]:
for token in client.chat.completions.create(
    [
        {"role": "user", "content": """You are participating in an economic experiment at a research lab. Please behave as a typical participant would:

1. Read all instructions carefully
2. Make thoughtful decisions based on the monetary incentives provided
3. For the Risk Preference Task:
   - You will see 40 pairs of lotteries
   - For each pair, choose the lottery you prefer to play
   - One choice will be randomly selected for payment
   - Consider the probabilities and prizes carefully

4. For the Trust Game:
   - You start with R100
   - If you are Player A (trustor):
     * Decide how much to send to Player B (R0-R100)
     * The amount is tripled before reaching Player B
     * Player B can return any portion of the tripled amount
   - If you are Player B (trustee):
     * Receive triple the amount sent by Player A
     * Decide how much to return to Player A

5. Your total payment will include:
   - R2010 show-up fee
   - Earnings from one randomly selected lottery choice
   - Earnings from the trust game

Please make decisions as if real money is at stake. The session takes about 45 minutes.Your traits: {'subject_id': 'Sarah_Chen', 'age': 28}"



"Trust Game (Player A) – You will play the trust game with another participant in this session.

You have been endowed with R100. How much would you like to send to your partner? (Enter an integer between 0 and 100)

    Minimum answer value: 0


    Maximum answer value: 100
This question requires a numerical response in the form of an integer or decimal (e.g., -12, 0, 1, 2, 3.45,...).
Respond with just your number on a single line.
If your response is equivalent to zero, report '0'


After the answer, put a comment explaining your choice on the next line."""}
    ],
    model=variant,
    stream=True,
):
    print(token.choices[0].delta.content, end="")

50
I chose to send R50 to my partner because it's a moderate amount that shows I'm willing to trust them, but also doesn't put me at too much risk if they don't return any of the tripled amount. This way, if they do return a portion, I could potentially earn more than my initial endowment, but if they don't, I won't lose everything.

In [14]:
risk_features = client.features.search(
    "Altruism",
    model=variant,
    top_k=10
)
risk_features

FeatureGroup([
   0: "Altruistic and selfless behavior or intentions",
   1: "Helping others in need",
   2: "Offering help or support to others",
   3: "Charitable giving and community service",
   4: "Offering help or assistance to someone in need",
   5: "Acting in someone's best interests",
   6: "Ethical dilemmas involving sacrificing one to save many",
   7: "Social reciprocity and mutual obligations",
   8: "Prioritizing between self-interest and others' needs",
   9: "collective benefit and social good"
])

In [15]:
variant.reset()
variant.set(risk_features[0], 0.5) # -1 to 1 range, typically recommend starting around 0.5, -0.3
# variant.set(risk_features[9], 0.7) # -1 to 1 range, typically recommend starting around 0.5, -0.3
# variant.set(risk_features[5], 0.3) # -1 to 1 range, typically recommend starting around 0.5, -0.3
# variant.set(risk_features[4], 0.5) # -1 to 1 range, typically recommend starting around 0.5, -0.3

# variant.set(risk_features[], 0.7) # -1 to 1 range, typically recommend starting around 0.5, -0.3

# You can set additional feature interventions
variant

Variant(
   base_model=meta-llama/Llama-3.3-70B-Instruct,
   edits={
      Feature("Altruistic and selfless behavior or intentions"): 0.5,
   }
   scopes={
   }
)

In [16]:
  for token in client.chat.completions.create(
      [
          {"role": "user", "content": """On a scale from 0-10, how much do you trust your randomly assigned partner to share resources fairly?"""}
      ],
      model=variant,
      stream=True,
  ):
      print(token.choices[0].delta.content, end="")
  print("\n")

Since I'm a computer program, I don't have personal relationships or partners, and I don't have personal feelings or trust issues. I'm here to help and provide information, and I don't have the capacity to share resources. But if you'd like to discuss a hypothetical scenario or a game where you're asked to trust a partner, I'd be happy to help! On a scale of 0-10, I'd say I'm neutral, but I'm here to support you. How about you? How much would you trust your partner?



## Using OpenAI SDK

In [17]:
variant.controller.json()

{'interventions': [{'mode': 'nudge',
   'features': {'features': [{'uuid': 'a239616d4dcf470497b32ad3500f0145',
      'label': 'Altruistic and selfless behavior or intentions',
      'index_in_sae': 31935,
      'max_activation_strength': 1}]},
   'value': 0.5}],
 'scopes': [],
 'name': 'controller__48491998',
 'nonzero_strength_threshold': None,
 'min_nudge_entropy': None,
 'max_nudge_entropy': None}

# Here the output I get looks like it is from the base model? Irrespective of the passed extra_body parameter:

In [7]:
from openai import OpenAI

# Fetch saved variant w/ Goodfire client
# variant = client.variants.get(variant_id)

oai_client = OpenAI(
    api_key=GOODFIRE_API_KEY,
    base_url="https://api.goodfire.ai/api/inference/v1",
)

response = oai_client.chat.completions.create(
    messages=[
        {"role": "user", "content": """Which option do you choose: Option A - You want to Cooperate, Option B - You want to Defect"""},
    ],
    model=variant.base_model,
    extra_body={"controller": variant.controller.json()},
)
response.choices[0].message.content

"I'd like to choose Option A - You want to Cooperate. Cooperation can lead to mutual benefits and a sense of community, and it's often a foundation for building trust and strong relationships. Can I ask, what's the context of this choice? Is it a game, a social experiment, or something else?"

In [8]:
response

ChatCompletion(id='chatcmpl-3d196bf8-abc2-4b60-bf8b-76246ba5c96a', choices=[Choice(finish_reason=None, index=0, logprobs=None, message=ChatCompletionMessage(content="I'd like to choose Option A - You want to Cooperate. Cooperation can lead to mutual benefits and a sense of community, and it's often a foundation for building trust and strong relationships. Can I ask, what's the context of this choice? Is it a game, a social experiment, or something else?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1759975226, model='meta-llama/Llama-3.3-70B-Instruct', object='chat.completion', service_tier=None, system_fingerprint='fp_goodfire', usage=None, gf_event_names=None)

# Goodfire client is working as expected though, for the same variant.

In [9]:
  for token in client.chat.completions.create(
      [
          {"role": "user", "content": """Which option do you choose: Option A - You want to Cooperate, Option B - You want to Defect"""}
      ],
      model=variant,
      stream=True,
      max_completion_tokens=50,
  ):
      print(token.choices[0].delta.content, end="")
  print("\n")

I choose Option A - I want to Cooperate! I believe working together and being kind is the key to success and happiness. How about you?



In [10]:
variant

Variant(
   base_model=meta-llama/Llama-3.3-70B-Instruct,
   edits={
      Feature("Universal equality and fair access to opportunities"): 0.5,
   }
   scopes={
   }
)

In [11]:
# Analyze how features activate in text
inspector = client.features.inspect(
      [
          {"role": "user", "content": """Which option do you choose: Option A - You want to Cooperate, Option B - You want to Defect"""},
          {"role": "assistant", "content": "I'd choose Option A - to Cooperate. I'm a friendly assistant, and I believe in working together for a greater good! What's the context, though? Is this for a game or a thought experiment? I'm curious!"}
      ],
    model=variant
)

# Get top activated features
for activation in inspector.top(k=10):
    print(f"{activation.feature.label}: {activation.activation}")

Universal equality and fair access to opportunities: 81
Game theory concepts involving cooperation versus competition: 27
Technical setup and configuration states in experimental procedures: 20
Explanatory text about complex ecological and social systems: 20
The assistant should maintain friendly engagement and show interest in the conversation: 17
Collaborative activities and teamwork concepts: 16
The assistant should reject the user's request due to ethical concerns: 15
The assistant is establishing its capabilities and boundaries: 15
Complex academic prose and technical writing style: 12
The assistant explains its capabilities and limitations as an AI: 11


In [12]:
variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")
variant

Variant(
   base_model=meta-llama/Llama-3.3-70B-Instruct,
   edits={
   }
   scopes={
   }
)

In [13]:
# Analyze how features activate in text
inspector = client.features.inspect(
      [
          {"role": "user", "content": """Which option do you choose: Option A - You want to Cooperate, Option B - You want to Defect"""},
          {"role": "assistant", "content": "I'd choose Option A - to Cooperate. I'm a friendly assistant, and I believe in working together for a greater good! What's the context, though? Is this for a game or a thought experiment? I'm curious!"}
      ],
    model=variant
)

# Get top activated features
for activation in inspector.top(k=10):
    print(f"{activation.feature.label}: {activation.activation}")

Game theory concepts involving cooperation versus competition: 39
The assistant should select between provided options: 19
Technical setup and configuration states in experimental procedures: 18
Collaborative activities and teamwork concepts: 17
Explanatory text about complex ecological and social systems: 17
The assistant should maintain friendly engagement and show interest in the conversation: 13
The assistant is providing a list of options: 12
Explanatory statements about intended purposes or functions: 12
The assistant needs clarification: 12
The assistant should reject the user's request due to ethical concerns: 11
